In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 02 — Data Cleaning and EDA\n",
    "**Stages 3 and 4: Clean → Explore**\n",
    "\n",
    "Cleans raw standings data then runs exploratory visualisations\n",
    "to understand the data before formal hypothesis testing."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.insert(0, '..')\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "from IPython.display import display\n",
    "from config import DATA_PROC_DIR\n",
    "from src.processors.cleaner import clean_standings\n",
    "from src.visualizers.eda_plots import (\n",
    "    plot_ga_distribution,\n",
    "    plot_correlation_heatmap,\n",
    "    plot_ga_vs_points_scatter,\n",
    ")\n",
    "%matplotlib inline\n",
    "plt.rcParams['figure.dpi'] = 120"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load raw data\n",
    "df_raw = pd.read_csv(DATA_PROC_DIR / 'standings_raw.csv')\n",
    "print(f'Loaded {len(df_raw)} rows')\n",
    "display(df_raw.describe())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# MISSING VALUE AUDIT — check before cleaning\n",
    "# ============================================================\n",
    "null_counts = df_raw.isnull().sum()\n",
    "null_pct    = (null_counts / len(df_raw) * 100).round(2)\n",
    "null_summary = pd.DataFrame({'count': null_counts, 'pct': null_pct})\n",
    "null_summary = null_summary[null_summary['count'] > 0]\n",
    "\n",
    "if len(null_summary) == 0:\n",
    "    print('No missing values in raw data')\n",
    "else:\n",
    "    print('Missing values found:')\n",
    "    display(null_summary)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# CLEAN DATA\n",
    "# ============================================================\n",
    "df_clean, quality_report = clean_standings(df_raw)\n",
    "\n",
    "print('Quality Report:')\n",
    "for key, val in quality_report.items():\n",
    "    print(f'  {key}: {val}')\n",
    "\n",
    "df_clean.to_csv(DATA_PROC_DIR / 'standings_clean.csv', index=False)\n",
    "print('Clean data saved')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# EDA Chart 1: Distribution\n",
    "fig = plot_ga_distribution(df_clean)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# EDA Chart 2: Correlation heatmap\n",
    "fig = plot_correlation_heatmap(df_clean)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# EDA Chart 3: GA vs Points\n",
    "fig = plot_ga_vs_points_scatter(df_clean)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ============================================================\n",
    "# OUTLIER REVIEW\n",
    "# ============================================================\n",
    "if 'goals_against_is_outlier' in df_clean.columns:\n",
    "    outliers = df_clean[df_clean['goals_against_is_outlier'] == True]\n",
    "    if len(outliers) > 0:\n",
    "        print(f'{len(outliers)} outliers flagged:')\n",
    "        display(outliers[['league_key','season_label','team_name',\n",
    "                           'position','goals_against','goals_against_zscore']])\n",
    "    else:\n",
    "        print('No outliers detected')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python", "version": "3.10.0"}
 },
 "nbformat": 4,
 "nbformat_minor": 4
}